In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [2]:
from data_load import scaled_data, raw_data, result, data_info

In [3]:
scaled_data.head()

,Unnamed: 0,용접 가압력,전류,전압,통전시간
0,0,0.004482,0.001717,0.002506,0.000461
1,1,0.005412,0.000501,0.002506,0.000922
2,2,0.004989,0.000358,0.507830,0.000922
3,3,0.005243,0.000358,0.002506,0.000922
4,4,0.005327,0.000143,0.002506,0.000461


In [4]:
raw_data.head()

,idx,Machine_Name,Item No,working time,Thickness 1(mm),Thickness 2(mm),weld force(bar),weld current(kA),weld Voltage(v),weld time(ms)
0,1,Spot-01,65235-25800,2020-03-24,0.7,0.7,2.33,14.57,2.701,72.0
1,2,Spot-01,65235-25800,2020-03-24,0.7,0.7,2.36,14.57,2.701,72.0
2,3,Spot-01,65235-25800,2020-03-24,0.7,0.7,2.37,14.54,2.703,71.0
3,4,Spot-01,65235-25800,2020-03-24,0.7,0.7,2.37,14.54,2.703,72.0
4,5,Spot-01,65235-25800,2020-03-24,0.7,0.7,2.36,14.56,2.704,72.0


In [ ]:
# Thickness는 모두 같은 값이므로 예측에 영향 X

print(np.sum(raw_data['Thickness 1(mm)'] != 0.7))
print(np.sum(raw_data['Thickness 2(mm)'] != 0.7))

0
0


In [12]:
scaled_data = scaled_data.drop(columns = ['Unnamed: 0'])

In [ ]:
model = IsolationForest(contamination=0.0038, random_state=42)

model.fit(scaled_data)

predictions = model.predict(scaled_data)


In [ ]:
idx = (predictions == -1)

X = scaled_data.loc[idx, :]

kmeans = KMeans(n_clusters=3, n_init='auto', random_state=42)

kmeans.fit(X)

labels = kmeans.labels_
print("각 데이터의 소속 그룹:", labels)

각 데이터의 소속 그룹: [1 0 0 0 0 0 0 1 1 1 0 2 2 0 0 0 0 0 0 0 0 1 1 1 0 0 2 2 0 1 0 0 0 0 0 2 1
 1 0 0 0 0 0 1 2 2]


In [ ]:
X['cluster'] = labels

X.head()

        용접 가압력        전류        전압      통전시간  cluster  pred
9     0.004989  0.000072  0.508875  0.000000        1     1
1882  0.061475  0.001645  0.000000  0.000461        0     0
1883  0.061475  0.002218  0.000000  0.000461        0     0
1884  0.061813  0.002218  0.000104  0.000461        0     0
1885  0.061813  0.002003  0.000104  0.000461        0     0
28 11 7


In [63]:
undercut_idx = result['defect'] == 1
lack_idx = result['defect'] == 2
crack_idx = result['defect'] == 3


anomaly1 = np.sum(labels == 0)
anomaly2 = np.sum(labels == 1)
anomaly3 = np.sum(labels == 2)
print(anomaly1, anomaly2, anomaly3)
print()

defect = pd.Series(result['defect'], dtype=int)
print(np.sum(defect[undercut_idx]), np.sum(defect[lack_idx]), np.sum(defect[crack_idx]))

28 11 7

9 18 12
